In [ ]:
import os
from pathlib import Path
from typing import Sequence
from tqdm import tqdm
import colormaps
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt
from functools import partial
import datetime
os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import DATADIR, DUNCANS_REGIONS_NAMES, FIGURES, YEARS, get_region, to_expr, squarify, polars_to_xarray
from jetutils.data import standardize, compute_anomalies_pl
from jetutils.jet_finding import find_all_jets, average_jet_categories, jet_position_as_da, track_jets, pers_from_cross_catd, spells_from_cross_catd_simple, to_one_large
from jetutils.plots import Clusterplot, COLORS, COLORS_EXT, STYLE_SHEET
from jetutils.anyspell import make_daily, mask_from_spells_pl, subset_around_onset
import altair as alt
alt.data_transformers.enable("vegafusion")

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/exp10")

In [ ]:
ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9])).filter(
    pl.col("time").dt.ordinal_day() > 166
)
summer = summer_filter["time"]
winter_filter = (
    ALL_TIMES
    .filter(pl.col("time").dt.month().is_in([12, 1, 2]))
)
winter = winter_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)
big_summer = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
big_summer_daily = big_summer.filter(big_summer["time"].dt.hour() == 0)

summer_doy = summer_daily.dt.ordinal_day().unique()
n_bootstraps = 100

jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets, int_EDJ_threshold=1.3e8)
props_uncat = pl.read_parquet(basepath.joinpath("props.parquet"))
props_uncat = props_uncat.with_columns(
    cs.exclude("time", "jet ID")
    .cast(pl.Float32())
    .replace((float("-inf"), float("inf"), float("nan")), None)
    .cast(pl.Float32())
)
props = squarify(average_jet_categories(props_uncat), ["time", "jet"])
phat_filter = (pl.col("is_polar") < 0.5) | (
    (pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8)
)
phat_props = props_uncat.filter(phat_filter)
index_columns = ["time", "jet ID"]
quants_I_want = ["hor", "t2m", "theta", "tp", "APVO", "CPVO", "PV330"]
for varname in tqdm(quants_I_want):
    factor = -1 if varname == "hor" else 1
    extra_filter = [pl.col(f"{varname}_interp") < 0] if varname == "hor" else []
    df = pl.scan_parquet(basepath.joinpath(f"{varname}_relative.parquet"))
    df_cold = df.filter(pl.col("n") >= 0, pl.col("n") < 5e5, *extra_filter).group_by("time", "jet ID").agg(pl.col(f"{varname}_interp").mean().alias(f"{varname}_cold") * factor).collect()
    df_warm = df.filter(pl.col("n") <= 0, pl.col("n") > -5e5, *extra_filter).group_by("time", "jet ID").agg(pl.col(f"{varname}_interp").mean().alias(f"{varname}_warm") * factor).collect()
    phat_props = phat_props.join(df_cold, on=["time", "jet ID"])
    phat_props = phat_props.join(df_warm, on=["time", "jet ID"])
phat_props = squarify(average_jet_categories(phat_props), ["time", "jet"])
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
cross = cross.filter(pl.col("jet ID") == pl.col("jet ID_right"))
cross = pers_from_cross_catd(cross)
pers = cross.select(
    "time",
    jet=pl.when(pl.col("jet ID") == 0).then(pl.lit("STJ")).otherwise(pl.lit("EDJ")),
    pers="pers",
)
phat_props = phat_props.join(pers, on=["time", "jet"], how="left")
phat_props = phat_props.with_columns(
    cs.exclude("time", "jet")
    .cast(pl.Float32())
    .replace((float("-inf"), float("inf"), float("nan")), None)
    .cast(pl.Float32())
)
phat_props = phat_props.join(
    phat_props.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))

phat_props_summer = summer_filter.join(phat_props, on="time")

spells_list = spells_from_cross_catd_simple(
    cross,
    season=summer,
    q_STJ=0.705,
    q_EDJ=0.836,
    minlen=datetime.timedelta(days=5),
    smooth=datetime.timedelta(hours=24),
    fill_holes=datetime.timedelta(hours=18),
)

for jet, spells in spells_list.items():
    print(jet, spells["spell"].n_unique())

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

In [ ]:
def bin_by(df: pl.DataFrame, by: str | pl.Expr, jet: str, data_vars: Sequence[str], n_q: int = 100):
    jet: pl.Expr = pl.col("jet") == jet
    qs = np.linspace(0, 1, n_q + 1).tolist()
    labels = [f"{int(q * 100):d}" for q in qs]
    by = to_expr(by)
    filter_ = by.qcut(qs[1:], labels=labels, allow_duplicates=True).cast(pl.String()).str.to_integer()
    filter_ = df.filter(jet).select("time", catd=filter_)
    aggs = {var: pl.col(var).mean() for var in data_vars}
    aggs = {"len": pl.col("time").len()} | aggs
    return (
        df
        .join(filter_, on="time", how="left")
        .group_by("catd", "jet")
        .agg(**aggs)
        .sort("catd", "jet")
    )

In [ ]:
plt.style.use(['seaborn-v0_8-darkgrid', STYLE_SHEET])

data_vars = [
    "mean_lat",
    "mean_theta",
    "mean_s",
    "tilt",
    "wavinessR16",
    "width",
]
cm = colormaps.bold
colors_props = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
cm = colormaps.pastel
colors_regions = cm(np.linspace(0, 1 - 1 / cm.N, cm.N) + 1 / 2 / cm.N)
summer = pl.col("time").dt.month().is_in([6, 7, 8])
winter = pl.col("time").dt.month().is_in([12, 1, 2])
spring = pl.col("time").dt.month().is_in([3, 4, 5])
autumn = pl.col("time").dt.month().is_in([9, 10, 11])
# phat_props = phat_props.with_columns(pl.col("dist").replace(np.inf, None))
props_as_df_anoms = compute_anomalies_pl(phat_props, standardize=True).filter(winter)
var_filters = ["mean_lat", "mean_s", "pers", "mean_lat", "mean_s", "pers"]
jet_filters = ["STJ", "STJ", "STJ", "EDJ", "EDJ", "EDJ"]

fig, axes = plt.subplots(len(var_filters), 2, sharex="all", sharey="all", constrained_layout=True, figsize=(11, len(var_filters) * 1.5))

for var_filter, jet_filter, axs in zip(var_filters, jet_filters, axes):
    huh = bin_by(props_as_df_anoms, var_filter, jet_filter, data_vars)
    for jet, ax in zip(["STJ", "EDJ"], axs[:2]):
        huh_ = huh.filter(pl.col("catd").is_not_null(), pl.col("jet") == jet)
        for var, color in zip(data_vars, colors_props):
            ax.plot(huh_["catd"], huh_[var], label=var, color=color, lw=2)
    # huh = bin_by(props_as_df_anoms, var_filter, jet_filter, [f"{var}_{reg}" for var in ["t2m", "tp"] for reg in range(1, 7)])
    # huh_ = huh.filter(pl.col("catd").is_not_null(), pl.col("jet") == "STJ")
    # for var, ax in zip(["t2m", "tp"], axs[2:]):
    #     for reg in range(1, 7):
    #         varname = f"{var}_{reg}"
    #         ax.plot(huh_["catd"], huh_[varname], label=DUNCANS_REGIONS_NAMES[reg - 1], color=colors_regions[reg - 1], lw=2)
    #     ax.set_ylim([-1, 1])
        
    ax.set_ylabel(f"{jet_filter} {var_filter}")
    ax.yaxis.set_label_position("right")

axes[0, 1].legend(ncol=2, fontsize=10, columnspacing=1, borderpad=0, borderaxespad=0.1, handletextpad=0.4, labelspacing=0.2)
# axes[0, 3].legend(ncol=2, fontsize=13, columnspacing=1, borderpad=0, borderaxespad=0.1, handletextpad=0.4, labelspacing=0.2)
axes[0, 0].set_title("STJ props")
axes[0, 1].set_title("EDJ props")
# axes[0, 2].set_title("Region. t2m")
# axes[0, 3].set_title("Region. tp")

fig.supxlabel("Quantile of predictor")
fig.supylabel("Std. Anomaly")

plt.show()
plt.style.use(['default', STYLE_SHEET])

In [ ]:
n_bins = 21
lat_bins = np.linspace(15, 81, n_bins)
lat_labels = ((lat_bins[1:] + lat_bins[:-1]) / 2).astype(str)
lat_cut = pl.col("mean_lat_EDJ").cut(lat_bins[1:-1], labels=lat_labels)
lat_cut = lat_cut.cast(pl.String()).cast(pl.Decimal(None, 4)) 
s_bins = np.linspace(0, 85, n_bins)
s_labels = ((s_bins[1:] + s_bins[:-1]) / 2).astype(str)
s_cut = pl.col("mean_s_EDJ").cut(s_bins[1:-1], labels=s_labels)
s_cut = s_cut.cast(pl.String()).cast(pl.Decimal(None, 4)) 
pivoted = winter_filter.join(phat_props, on="time").pivot("jet", index="time")
cc = polars_to_xarray(pivoted.with_columns(lat_cut=lat_cut, s_cut=s_cut).group_by("lat_cut", "s_cut").agg(cs.float().mean(), cs.integer().mean(), pop=pl.len()).drop_nulls(["lat_cut", "s_cut"]), ["lat_cut", "s_cut"])
cc = cc.assign_coords(lat_cut=cc.lat_cut.astype("float"), s_cut=cc.s_cut.astype("float"))

In [ ]:
cc["pop"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["mean_lat_STJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["lon_ext_STJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["hor_cold_STJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["PV330_cold_STJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["tp_cold_EDJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["tp_warm_EDJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["t2m_cold_EDJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["hor_warm_STJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["theta_warm_EDJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["theta_cold_EDJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["theta_cold_STJ"].plot()

In [ ]:
cc.where(cc["pop"] > 100)["mean_lat_var_STJ"].plot(vmax=8)

In [ ]:
cc.where(cc["pop"] > 100)["pers_EDJ"].plot(vmax=None)

In [ ]:
cc.where(cc["pop"] > 100)["pers_STJ"].plot(vmax=None)